In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "pyproject.toml").is_file()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.load_dataset import laod_dataframes

from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from surprise import SVD, Dataset, Reader

In [ ]:
import joblib

ARTIFACTS_PATH = PROJECT_ROOT / "data" / "procced" / "recommender.joblib"
ARTIFACTS = None
REQUIRED_ARTIFACTS = {
    "train_df", "test_df", "df_books", "user_history", "test_relevant",
    "popular_books", "positive_history", "books_content", "tfidf_matrix",
    "book_id_to_idx", "interaction_matrix", "item_similarity",
    "user_to_idx", "book_ids", "svd_model",
}


def _validate_artifacts(artifacts):
    missing = REQUIRED_ARTIFACTS.difference(artifacts)
    if missing:
        raise ValueError(f"Не хватает артефактов: {', '.join(sorted(missing))}")


def save_artifacts(**artifacts):
    global ARTIFACTS
    _validate_artifacts(artifacts)
    ARTIFACTS_PATH.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = ARTIFACTS_PATH.with_suffix(".joblib.tmp")
    joblib.dump(artifacts, temporary_path)
    temporary_path.replace(ARTIFACTS_PATH)
    ARTIFACTS = None
    return ARTIFACTS_PATH


def load_artifacts(*, reload=False):
    global ARTIFACTS
    if ARTIFACTS is None or reload:
        if not ARTIFACTS_PATH.is_file():
            raise FileNotFoundError(
                f"Артефакты не найдены: {ARTIFACTS_PATH}. "
                "Выполните обучение и ячейку save_artifacts."
            )
        artifacts = joblib.load(ARTIFACTS_PATH)
        _validate_artifacts(artifacts)
        ARTIFACTS = artifacts
    return ARTIFACTS


## 1. Загрузка данных

In [2]:
dataset_dict = laod_dataframes()

df_books = dataset_dict["df_books"]
df_ratings = dataset_dict["df_ratings"]
df_book_tags = dataset_dict["df_book_tags"]
df_tags = dataset_dict["df_tags"]
df_to_read = dataset_dict["df_to_read"]

### 1.1 Проверка данных

In [3]:
for name, df in dataset_dict.items():
    print(f"{name}: {df.shape}")
    print(df.columns.tolist())
    print("-" * 50)

df_books: (10000, 23)
['id', 'book_id', 'best_book_id', 'work_id', 'books_count', 'isbn', 'isbn13', 'authors', 'original_publication_year', 'original_title', 'title', 'language_code', 'average_rating', 'ratings_count', 'work_ratings_count', 'work_text_reviews_count', 'ratings_1', 'ratings_2', 'ratings_3', 'ratings_4', 'ratings_5', 'image_url', 'small_image_url']
--------------------------------------------------
df_ratings: (981756, 3)
['book_id', 'user_id', 'rating']
--------------------------------------------------
df_book_tags: (999912, 3)
['goodreads_book_id', 'tag_id', 'count']
--------------------------------------------------
df_tags: (34252, 2)
['tag_id', 'tag_name']
--------------------------------------------------
df_to_read: (912705, 2)
['user_id', 'book_id']
--------------------------------------------------


### 1.2 Очистка данных

In [4]:
def clean_ratings(df_ratings, df_books):
    """Отчистка рейтингов"""
    ratings = df_ratings[["user_id", "book_id", "rating"]].copy()

    ratings = ratings.dropna()
    ratings = ratings[ratings["book_id"].isin(df_books["book_id"])]
    ratings = ratings.groupby(['user_id','book_id' ], as_index=False)['rating'].agg('mean')
    # Оставляем книги, которые присутствуют в каталоге
    ratings = ratings[ratings["book_id"].isin(df_books["id"])]

    return ratings.reset_index(drop=True)


ratings = clean_ratings(df_ratings, df_books)
print("Количество взаимодействий:", len(ratings))
print("Пользователей:", ratings["user_id"].nunique())
print("Книг:", ratings["book_id"].nunique())

Количество взаимодействий: 79531
Пользователей: 28906
Книг: 812


### 1.3. Разделение на train и test

In [5]:

def make_train_test_split(
    ratings: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_parts = []
    test_parts = []
    """Деление на обучающю и тестовую выборку"""

    for user_id, group in ratings.groupby("user_id"):
        if len(group) < 5:
            train_parts.append(group)
            continue

        train_group, test_group = train_test_split(
            group,
            test_size=0.2,
            random_state=RANDOM_STATE,
            shuffle=False
        )
        train_parts.append(train_group)
        test_parts.append(test_group)

    train_df = pd.concat(train_parts, ignore_index=True)
    test_df = (
        pd.concat(test_parts, ignore_index=True)
        if test_parts
        else ratings.iloc[0:0].copy()
    )
    return train_df, test_df


train_df, test_df = make_train_test_split(ratings)

print("Train:", train_df.shape)
print("Test:", test_df.shape)



Train: (70551, 3)
Test: (8980, 3)


### 1.4 Создадим историю пользователей:

In [6]:
def make_user_history(train_df:pd.DataFrame)-> dict:
    """История пользователя"""
    user_history = (train_df.groupby("user_id")["book_id"].agg(set).to_dict())

    return user_history

def make_relevant(user_history:pd.DataFrame)-> dict:
    """Ищем пользователей с реелвантыми книгами"""
    # Только оценки 4 и 5 считаем релевантными
    test_relevant = (test_df[test_df["rating"] >= 4].groupby("user_id")["book_id"].agg(set).to_dict())

    return test_relevant


user_history = make_user_history(train_df)
test_relevant = make_relevant(train_df)

print("История пользователя",len(user_history))
print("Пользователей с релевантными книгами:",len(test_relevant))



История пользователя 28906
Пользователей с релевантными книгами: 3667


### 1.5 Модель популярности (Popularity-Based)

In [7]:

def make_popular_books(train_df: pd.DataFrame)->pd.DataFrame:

    book_popularity = (train_df.groupby("book_id").agg(
            mean_rating=("rating", "mean"),
            ratings_count=("rating", "count")
        )
        .reset_index()
    )

    # Минимальное число оценок для оценки популярности
    min_ratings = 50

    popular_books = (
        book_popularity[
            book_popularity["ratings_count"] >= min_ratings
        ]
        .sort_values(
            ["mean_rating", "ratings_count"],
            ascending=False
        )
    )
    return popular_books

popular_books = make_popular_books(train_df)

popular_books.head()


,book_id,mean_rating,ratings_count
769,9566,4.743902,82
354,4708,4.659574,94
771,9569,4.642857,84
448,5344,4.565217,92
762,9531,4.544118,68


### 1.6 Функция рекомендации

In [ ]:
def recommend_popular(user_id, N=10):
    if N <= 0:
        return []
    seen = user_history.get(user_id, set())
    recommendations = popular_books[
        ~popular_books["book_id"].isin(seen)
    ].head(N)
    return [
        {"book_id": int(book_id), "score": float(score)}
        for book_id, score in zip(
            recommendations["book_id"], recommendations["mean_rating"]
        )
    ]


### 1.7 Контентная модель (Content-Based Filtering)

In [9]:
def make_discription(df_book_tags:pd.DataFrame)->pd.DataFrame:
    """Собираем теги для частотного кодирования"""
    # 1. Объединяем теги с их названиями
    book_tags = df_book_tags.merge(
        df_tags,
        on="tag_id",
        how="left"
    )

    # 2. Выбираем 10 наиболее значимых тегов для каждой книги
    book_tags = (
        book_tags
        .sort_values("count", ascending=False)
        .groupby("goodreads_book_id")
        .head(10)
        .copy()
    )

    # 3. Подготавливаем названия тегов
    book_tags["tag_name"] = (
        book_tags["tag_name"]
        .fillna("")
        .str.replace("-", " ", regex=False)
    )

    # 4. Объединяем теги в одну строку для каждой книги
    tags_aggregated = (
        book_tags.groupby("goodreads_book_id")["tag_name"]
        .agg(" ".join)
        .reset_index()
    )

    # 5. Создаём таблицу с текстовым описанием книг
    books_content = df_books[
        ["id", "book_id", "title", "original_title"]
    ].copy()

    # 6. Объединяем книги с тегами по разным названиям ключа
    books_content = books_content.merge(
        tags_aggregated,
        left_on="book_id",
        right_on="goodreads_book_id",
        how="left"
    )

    # 7. Формируем текст для TF-IDF
    books_content["content"] = (
        books_content["original_title"]
        .fillna(books_content["title"])
        .fillna("")
        + " "
        + books_content["tag_name"].fillna("")
    )
    return books_content

books_content = make_discription(df_book_tags)

books_content[["book_id", "content"]].head()


,book_id,content
0,2767052,The Hunger Games favorites currently reading y...
1,3,Harry Potter and the Philosopher's Stone to re...
2,41865,Twilight young adult fantasy favorites vampire...
3,2657,To Kill a Mockingbird classics favorites to re...
4,4671,The Great Gatsby classics favorites fiction cl...


In [10]:
def make_vectorization(books_content:pd.DataFrame):
    """Делаем векторизацию"""
    tfidf = TfidfVectorizer(
        stop_words="english",
        max_features=20000,
        ngram_range=(1, 2)
    )

    return tfidf.fit_transform(
        books_content["content"]
    )

tfidf_matrix = make_vectorization(books_content)
print("Размер TF-IDF матрицы:", tfidf_matrix.shape)

Размер TF-IDF матрицы: (10000, 20000)


In [ ]:
book_id_to_idx = {
    int(book_id): idx
    for idx, book_id in enumerate(books_content["book_id"])
}

def get_similar_books(book_id, tfidf_matrix, N=10):
    if book_id not in book_id_to_idx:
        return []

    idx = int(book_id_to_idx[book_id])

    similarities = cosine_similarity(
        tfidf_matrix[idx],
        tfidf_matrix
    ).ravel()

    similarities[idx] = -1
    top_indices = np.argsort(similarities)[::-1][:N]

    return (
        books_content.iloc[top_indices]["book_id"]
        .tolist()
    )


#test
book_id = 1

similar_ids = get_similar_books(book_id,tfidf_matrix, N=5)

df_books[
    df_books["book_id"].isin(similar_ids)
][["book_id", "title"]]

### 1.8 Персональные контентные рекомендации

In [ ]:
positive_history = (
    train_df[train_df["rating"] >= 4]
    .groupby("user_id")["book_id"]
    .agg(list)
    .to_dict()
)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
import numpy as np


def recommend_content(user_id, N=10):
    if N <= 0:
        return []
    liked_books = positive_history.get(user_id, [])
    indices = [
        book_id_to_idx[book_id]
        for book_id in liked_books
        if book_id in book_id_to_idx
    ]
    if not indices:
        return recommend_popular(user_id, N)

    user_profile = np.asarray(tfidf_matrix[indices].mean(axis=0))
    user_profile = normalize(user_profile)
    scores = cosine_similarity(user_profile, tfidf_matrix).ravel()

    seen = user_history.get(user_id, set())
    for book_id in seen:
        idx = book_id_to_idx.get(book_id)
        if idx is not None:
            scores[idx] = -np.inf

    top_indices = np.argsort(scores)[::-1][:N]
    return [
        {
            "book_id": int(books_content.iloc[idx]["book_id"]),
            "score": float(scores[idx]),
        }
        for idx in top_indices
        if np.isfinite(scores[idx])
    ]


### 1.9 Коллаборативная фильтрация (Item-Based CF)

In [13]:

from scipy.sparse import csr_matrix

def make_interaction_matrix(train_df, df_books):
    user_ids = np.sort(train_df["user_id"].unique())
    book_ids = np.sort(df_books["book_id"].unique())

    user_to_idx = {
        user_id: idx
        for idx, user_id in enumerate(user_ids)
    }

    book_to_idx = {
        book_id: idx
        for idx, book_id in enumerate(book_ids)
    }

    rows = train_df["user_id"].map(user_to_idx).to_numpy()
    cols = train_df["book_id"].map(book_to_idx).to_numpy()

    csr =  csr_matrix(
        (
            train_df["rating"].to_numpy(dtype=np.float32),
            (rows, cols)
        ),
        shape=(len(user_ids), len(book_ids)),
        dtype=np.float32 ,
    )
    return csr, user_to_idx, book_ids

interaction_matrix, user_to_idx, book_ids = make_interaction_matrix(train_df, df_books)

print("Матрица взаимодействий:", interaction_matrix.shape)


Матрица взаимодействий: (28906, 10000)


Вычислим матрицу сходства

In [14]:

from sklearn.metrics.pairwise import cosine_similarity

item_matrix = interaction_matrix.T.tocsr()

item_similarity = cosine_similarity(
    item_matrix,
    dense_output=False
)

item_similarity.setdiag(0)
item_similarity.eliminate_zeros()

print("Матрица сходства:", item_similarity.shape)


Матрица сходства: (10000, 10000)


In [ ]:
def recommend_item_cf(user_id, N=10):
    if N <= 0:
        return []
    if user_id not in user_to_idx:
        return recommend_popular(user_id, N)

    user_idx = user_to_idx[user_id]
    user_ratings = interaction_matrix.getrow(user_idx)
    weighted_scores = (item_similarity @ user_ratings.T).toarray().ravel()

    observed = user_ratings.copy()
    observed.data = np.ones_like(observed.data)
    normalizer = (item_similarity @ observed.T).toarray().ravel()
    scores = np.divide(
        weighted_scores,
        normalizer,
        out=np.zeros_like(weighted_scores),
        where=normalizer > 0,
    )
    scores[normalizer <= 0] = -np.inf
    scores[user_ratings.indices] = -np.inf

    valid_indices = np.where(np.isfinite(scores))[0]
    top_indices = valid_indices[
        np.argsort(scores[valid_indices])[::-1][:N]
    ]
    return [
        {"book_id": int(book_ids[idx]), "score": float(scores[idx])}
        for idx in top_indices
    ]


### 1.20 Обучение SVD

In [16]:


reader = Reader(rating_scale=(1, 5))

dataset = Dataset.load_from_df(
    train_df[["user_id", "book_id", "rating"]],
    reader
)

trainset = dataset.build_full_trainset()

svd_model = SVD(
    n_factors=50,
    n_epochs=30,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)

svd_model.fit(trainset)

prediction = svd_model.predict(
    uid=1,
    iid=10
)

print("Предсказанная оценка:", prediction.est)

Предсказанная оценка: 4.135618058786348


In [ ]:
def recommend_svd(user_id, N=10):
    if N <= 0:
        return []
    seen = user_history.get(user_id, set())
    candidates = [
        int(book_id)
        for book_id in book_ids
        if book_id not in seen
    ]
    predictions = [
        {
            "book_id": book_id,
            "score": float(svd_model.predict(uid=int(user_id), iid=book_id).est),
        }
        for book_id in candidates
    ]
    predictions.sort(key=lambda x: x["score"], reverse=True)
    return predictions[:N]


In [ ]:
# Выполнить после обучения. Данные сохраняются в data/procced/recommender.joblib.
artifacts_path = save_artifacts(
    train_df=train_df,
    test_df=test_df,
    df_books=df_books,
    user_history=user_history,
    test_relevant=test_relevant,
    popular_books=popular_books,
    positive_history=positive_history,
    books_content=books_content,
    tfidf_matrix=tfidf_matrix,
    book_id_to_idx=book_id_to_idx,
    interaction_matrix=interaction_matrix,
    item_similarity=item_similarity,
    user_to_idx=user_to_idx,
    book_ids=book_ids,
    svd_model=svd_model,
)
print("Артефакты сохранены:", artifacts_path)

# Для нового сеанса загрузите артефакты в глобальные переменные:
# globals().update(load_artifacts())


## 2. Построение гибридной рекомендательной системы

In [ ]:
models = {
    "Popularity": recommend_popular,
    "Content-Based": recommend_content,
    "Item-Based CF": recommend_item_cf,
    "SVD": recommend_svd,
}

user_id = 1

for model_name, recommend_fn in models.items():
    print(f"\n{model_name}")
    recommendations = recommend_fn(user_id, N=5)
    display(pd.DataFrame(recommendations, columns=["book_id", "score"]))


### Использование функций рекомендаций

Функции определены в ноутбуке и используют подготовленные глобальные переменные.
У всех одинаковая сигнатура `(user_id, N=10)` и результат — список словарей
`{"book_id": ..., "score": ...}`. Ячейка `models` выводит таблицу каждой модели.

После перезапуска можно восстановить данные без обучения: выполните первую
ячейку, ячейку с `load_artifacts`, затем `globals().update(load_artifacts())`
и ячейки с определениями четырёх рекомендательных функций.

```python
globals().update(load_artifacts())
recommendations = recommend_content(user_id=1, N=5)
display(pd.DataFrame(recommendations, columns=["book_id", "score"]))
```
